In [12]:
import os, requests
from google.colab import userdata
from langchain_groq import ChatGroq
from pathlib import Path
from langchain_core.prompts import HumanMessagePromptTemplate, AIMessagePromptTemplate, SystemMessagePromptTemplate, ChatPromptTemplate, FewShotChatMessagePromptTemplate


In [2]:
def _read_env_file(name, filename="secrets.env"):
    """Look for KEY=value in secrets.env in the working dir or /content."""
    for folder in (Path.cwd(), Path("/content")):
        path = folder / filename
        if path.is_file():
            for line in path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, _, val = line.partition("=")
                if key.strip().removeprefix("export ").strip() == name:
                    return val.strip().strip('"').strip("'")
    return None

def get_secret(name, required=True):
    value = None
    try:
        value = userdata.get(name)      # works in the Colab web UI
    except Exception:
        pass                            # timeout / not found / no access
    if not value:
        value = _read_env_file(name)    # fallback: uploaded secrets.env
    if value:
        os.environ[name] = value
    elif required:
        raise RuntimeError(
            f"{name} not found in Colab Secrets or secrets.env. "
            "Upload secrets.env to the Colab server and re-run."
        )
    return value

groq_key = get_secret("GROQ_API_KEY")


print("GROQ key loaded:", bool(groq_key))


GROQ key loaded: True


In [4]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    model_kwargs= {"seed" : 350 },
    temperature= 0,
    max_tokens= 250
)

In [5]:
template_s = 'you will be given a 6 digit number , break it down into 3 digits and categorize them accordingly in English words'
template_h = '{input_digit}'
template_Ai = '{description}'

In [8]:
template_system = SystemMessagePromptTemplate.from_template(template =template_s)
template_human = HumanMessagePromptTemplate.from_template(template = template_h)
template_Ai = AIMessagePromptTemplate.from_template(template =template_Ai)

In [10]:
chat_template = ChatPromptTemplate.from_messages([template_system, template_human, template_Ai])

In [20]:
examples = [
    {
        'input_digit' : '402196',
        'description': '''
        course no. = four hundred and two (402)
        student id = one hundred and ninety six (196)
        '''
    },
    {
        'input_digit' : '203653',
                'description': '''
                course no. = two hundred and three (203)
                student id = six hundred and fifty three (653)
                '''
    },
    {
        'input_digit' : '192246',
                        'description': '''
                        course no. = one hundred and ninety two (192)
                        student id = two hundred and forty six (246)
                        '''
    }
]

In [21]:
few_shot_template = FewShotChatMessagePromptTemplate(
        examples = examples, 
        example_prompt = chat_template,
        input_variables = ['input_digit']
)

In [22]:
main_chat_template = ChatPromptTemplate.from_messages([few_shot_template, template_h])

In [33]:
chat_values = main_chat_template.invoke({'input_digit': '476963'})

print(chat_values)

messages=[SystemMessage(content='you will be given a 6 digit number , break it down into 3 digits and categorize them accordingly in English words', additional_kwargs={}, response_metadata={}), HumanMessage(content='402196', additional_kwargs={}, response_metadata={}), AIMessage(content='\n        course no. = four hundred and two (402)\n        student id = one hundred and ninety six (196)\n        ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), SystemMessage(content='you will be given a 6 digit number , break it down into 3 digits and categorize them accordingly in English words', additional_kwargs={}, response_metadata={}), HumanMessage(content='203653', additional_kwargs={}, response_metadata={}), AIMessage(content='\n                course no. = two hundred and three (203)\n                student id = six hundred and fifty three (653)\n                ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), System

In [34]:
response = llm.invoke(chat_values)

print(response.content)

        course no. = four hundred and seventy six (476)  
        student id = nine hundred and sixty three (963)
